# ⚡ سيرفر التبييض السريع بالـ GPU (Google Colab T4)
شغّل الخلية دي بس وهتطلعلك لينك `WORKER_URL` جاهز تحطه في Railway عشان الفصل يتبيّض في ثواني بالـ GPU.

In [ ]:
import os, time, re, subprocess

# 1. ضع نفس التوكن الموجود في Railway هنا:
WORKER_TOKEN = 'put-your-worker-token-here'
REPO_URL = ''  # اختياري: لو عايز تسحب الملفات، أو ارفع ملفات البوت في فولدر Colab

!pip install -q fastapi 'uvicorn[standard]' python-multipart aiohttp opencv-python-headless numpy onnxruntime-gpu Pillow simple-lama-inpainting
!wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /content/cloudflared.deb > /dev/null

os.environ['WORKER_TOKEN'] = WORKER_TOKEN
os.environ['WORKER_ENGINE'] = 'builtin'
os.environ['PAGE_CONCURRENCY'] = '2'
os.environ['CV_THREADS'] = '4'

subprocess.Popen(['uvicorn', 'worker_server:app', '--host', '127.0.0.1', '--port', '8001'], stdout=open('/content/worker.log', 'w'), stderr=subprocess.STDOUT)
subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8001'], stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)

time.sleep(8)
log_txt = open('/content/tunnel.log').read()
urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log_txt)
print('✅ WORKER_URL =', urls[0] if urls else 'Wait a few seconds and check /content/tunnel.log')
